In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.utils import to_categorical
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import os


In [2]:
BASE = r"D:\SER_Cross\features"

X_train = np.load(os.path.join(BASE, "train", "X.npy"))
y_train = np.load(os.path.join(BASE, "train", "y.npy"))

X_val = np.load(os.path.join(BASE, "val", "X.npy"))
y_val = np.load(os.path.join(BASE, "val", "y.npy"))

X_test = np.load(os.path.join(BASE, "test", "X.npy"))
y_test = np.load(os.path.join(BASE, "test", "y.npy"))

print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)


(5068, 128, 300) (5068,)
(1031, 128, 300) (1031,)
(1128, 128, 300) (1128,)


In [3]:
X_train = X_train[..., np.newaxis]
X_val   = X_val[..., np.newaxis]
X_test  = X_test[..., np.newaxis]

print(X_train.shape)


(5068, 128, 300, 1)


In [4]:
NUM_CLASSES = 5

y_train_cat = to_categorical(y_train, NUM_CLASSES)
y_val_cat   = to_categorical(y_val, NUM_CLASSES)
y_test_cat  = to_categorical(y_test, NUM_CLASSES)


In [5]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weights = dict(enumerate(class_weights))
print(class_weights)


{0: 1.0602510460251047, 1: 0.9859922178988327, 2: 0.9859922178988327, 3: 0.9859922178988327, 4: 0.9859922178988327}


In [6]:
class Attention(layers.Layer):
    def __init__(self):
        super().__init__()

    def call(self, inputs):
        # inputs: (batch, time, features)
        score = tf.nn.softmax(tf.reduce_mean(inputs, axis=-1), axis=1)
        score = tf.expand_dims(score, axis=-1)
        context = tf.reduce_sum(inputs * score, axis=1)
        return context


In [7]:
input_shape = (128, 300, 1)

inputs = layers.Input(shape=input_shape)

# CNN block
x = layers.Conv2D(32, (3, 3), activation="relu", padding="same")(inputs)
x = layers.MaxPooling2D((2, 2))(x)
x = layers.BatchNormalization()(x)

x = layers.Conv2D(64, (3, 3), activation="relu", padding="same")(x)
x = layers.MaxPooling2D((2, 2))(x)
x = layers.BatchNormalization()(x)

# reshape for LSTM
x = layers.Reshape((x.shape[1], x.shape[2] * x.shape[3]))(x)

# BiLSTM
x = layers.Bidirectional(layers.LSTM(128, return_sequences=True))(x)

# Attention
x = Attention()(x)

# Dense
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = models.Model(inputs, outputs)
model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 128, 300, 1)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 128, 300, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 150, 32)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64, 150, 32)    │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 64, 150, 64)    │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 75, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 32, 75, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 32, 4800)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 32, 256)        │     5,047,296 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ attention (Attention)           │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,100,037 (19.46 MB)

 Trainable params: 5,099,845 (19.45 MB)

 Non-trainable params: 192 (768.00 B)

In [8]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)


In [9]:
callbacks = [
    EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
    ModelCheckpoint(
        "best_ser_model.h5",
        monitor="val_accuracy",
        save_best_only=True
    )
]


In [10]:
history = model.fit(
    X_train, y_train_cat,
    validation_data=(X_val, y_val_cat),
    epochs=40,
    batch_size=32,
    class_weight=class_weights,
    callbacks=callbacks
)


Epoch 1/40
159/159 ━━━━━━━━━━━━━━━━━━━━ 0s 419ms/step - accuracy: 0.3165 - loss: 1.5366

159/159 ━━━━━━━━━━━━━━━━━━━━ 73s 438ms/step - accuracy: 0.3366 - loss: 1.4959 - val_accuracy: 0.3201 - val_loss: 1.5192
Epoch 2/40
159/159 ━━━━━━━━━━━━━━━━━━━━ 0s 418ms/step - accuracy: 0.3963 - loss: 1.3968

159/159 ━━━━━━━━━━━━━━━━━━━━ 69s 435ms/step - accuracy: 0.4002 - loss: 1.3871 - val_accuracy: 0.3608 - val_loss: 1.4639
Epoch 3/40
159/159 ━━━━━━━━━━━━━━━━━━━━ 0s 406ms/step - accuracy: 0.4479 - loss: 1.3044

159/159 ━━━━━━━━━━━━━━━━━━━━ 67s 422ms/step - accuracy: 0.4390 - loss: 1.3151 - val_accuracy: 0.3812 - val_loss: 1.4245
Epoch 4/40
159/159 ━━━━━━━━━━━━━━━━━━━━ 0s 406ms/step - accuracy: 0.4576 - loss: 1.2589

159/159 ━━━━━━━━━━━━━━━━━━━━ 67s 423ms/step - accuracy: 0.4676 - loss: 1.2478 - val_accuracy: 0.4132 - val_loss: 1.3698
Epoch 5/40
159/159 ━━━━━━━━━━━━━━━━━━━━ 0s 407ms/step - accuracy: 0.5023 - loss: 1.1741

159/159 ━━━━━━━━━━━━━━━━━━━━ 67s 424ms/step - accuracy: 0.5071 - loss: 1.1715 - val_accuracy: 0.4200 - val_loss: 1.3476
Epoch 6/40
159/159 ━━━━━━━━━━━━━━━━━━━━ 0s 412ms/step - accuracy: 0.5654 - loss: 1.0857

159/159 ━━━━━━━━━━━━━━━━━━━━ 68s 429ms/step - accuracy: 0.5568 - loss: 1.0940 - val_accuracy: 0.4277 - val_loss: 1.3794
Epoch 7/40
159/159 ━━━━━━━━━━━━━━━━━━━━ 0s 409ms/step - accuracy: 0.6207 - loss: 0.9766

159/159 ━━━━━━━━━━━━━━━━━━━━ 68s 425ms/step - accuracy: 0.6077 - loss: 0.9939 - val_accuracy: 0.4374 - val_loss: 1.3513
Epoch 8/40
159/159 ━━━━━━━━━━━━━━━━━━━━ 0s 405ms/step - accuracy: 0.6789 - loss: 0.8671

159/159 ━━━━━━━━━━━━━━━━━━━━ 67s 422ms/step - accuracy: 0.6766 - loss: 0.8693 - val_accuracy: 0.4510 - val_loss: 1.3672
Epoch 9/40
159/159 ━━━━━━━━━━━━━━━━━━━━ 67s 423ms/step - accuracy: 0.7273 - loss: 0.7421 - val_accuracy: 0.4287 - val_loss: 1.4525
Epoch 10/40
159/159 ━━━━━━━━━━━━━━━━━━━━ 68s 427ms/step - accuracy: 0.7991 - loss: 0.5839 - val_accuracy: 0.4491 - val_loss: 1.4427
Epoch 11/40
159/159 ━━━━━━━━━━━━━━━━━━━━ 0s 408ms/step - accuracy: 0.8764 - loss: 0.4273

159/159 ━━━━━━━━━━━━━━━━━━━━ 67s 424ms/step - accuracy: 0.8710 - loss: 0.4317 - val_accuracy: 0.4656 - val_loss: 1.4991


In [11]:
test_loss, test_acc = model.evaluate(X_test, y_test_cat)
print("Test Accuracy:", test_acc)


36/36 ━━━━━━━━━━━━━━━━━━━━ 3s 79ms/step - accuracy: 0.4690 - loss: 1.2057
Test Accuracy: 0.4689716398715973


In [12]:
y_pred = model.predict(X_test)
y_pred_labels = np.argmax(y_pred, axis=1)

print(classification_report(
    y_test,
    y_pred_labels,
    target_names=["neutral", "happy", "sad", "angry", "fear"]
))


36/36 ━━━━━━━━━━━━━━━━━━━━ 3s 83ms/step
              precision    recall  f1-score   support

     neutral       0.41      0.39      0.40       216
       happy       0.38      0.44      0.40       228
         sad       0.57      0.48      0.52       228
       angry       0.61      0.68      0.65       228
        fear       0.37      0.35      0.36       228

    accuracy                           0.47      1128
   macro avg       0.47      0.47      0.47      1128
weighted avg       0.47      0.47      0.47      1128

